# Music Review Insight Finder: Expanded Evaluation

This expanded study uses a more expaned dataset with the purpose of attempting to improve model efficacy, following the last few notebooks that confirmed the original model's functionality. The expanded dataset used 400 training passages, 200 validation passages, and 200 test passages. One fresh three-epoch LoRA run used the established B configuration. Validation selected the checkpoint, and one final paired test compared the unadapted and adapted models.

## 1. Model and Environment

The study used `Qwen/Qwen2.5-0.5B-Instruct`, pinned to revision `7ae557604adf67be50417f59c2c2f167def9a775`. The random seed was 1509509.

Training ran on a Tesla T4 in Google Colab with runtime version 2026.07. Package versions were pinned to support reproducibility.

In [1]:
# Prepared defaults block every cloud/data/model action.
EXPECTED_FREEZE_SHA256 = '8332b6ad842b1608fe760af1aaf9f13beae9de9ed6ebddadd1cf9aadaff342a5'  # verified identity only; Aaron authorized these stages; separate test/preflight gates still apply
ALLOW_CLOUD_SETUP = True
ALLOW_DATA_UPLOAD = True
ALLOW_TRAINING = True
ALLOW_FINAL_TEST = True
PINS = {"transformers":"4.57.1", "peft":"0.17.1", "accelerate":"1.10.1", "safetensors":"0.6.2",
        "numpy":"1.26.4", "pandas":"2.2.3", "matplotlib":"3.9.4", "scikit-learn":"1.6.1"}
def authorized(flag):
    assert flag is True, "This stage remains disabled until Aaron authorizes the exact reviewed freeze."
    assert len(EXPECTED_FREEZE_SHA256)==64 and all(c in "0123456789abcdef" for c in EXPECTED_FREEZE_SHA256), "Paste the exact authorized freeze SHA-256."
print("Prepared configuration only. No data transferred or models loaded.")

Prepared configuration only. No data transferred or models loaded.


In [2]:
authorized(ALLOW_CLOUD_SETUP)
import os,sys,importlib.metadata,subprocess
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
os.environ["WANDB_DISABLED"]="true"
assert (3,10)<=sys.version_info[:2]<=(3,12), "Use the known Python3.12 runtime."
import torch
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "Select the authorized free T4 runtime."
assert torch.__version__.split("+")[0]=="2.11.0", "Use the known working runtime2026.07 build."
def version(name):
    try:return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:return None
SETUP_NEEDS_RESTART=False
if any(version(n)!=v for n,v in PINS.items()):
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet"]+[n+"=="+v for n,v in PINS.items()])
    SETUP_NEEDS_RESTART=True
    print("Restart the session now. Then rerun the configuration cell with authorized values before continuing.")
else: print("Known pinned environment matches; no installation needed.")

Known pinned environment matches; no installation needed.


## 2. Data Preparation and Evaluation Splits

The expanded dataset contained the following passages and review/release groups:

| Split | Passages | Groups |
|---|---:|---:|
| Training | 400 | 180 |
| Validation | 200 | 99 |
| Test | 200 | 99 |

The original 140 training rows were retained without changes to their values or provenance. The 660 new usable labels received recorded human review after AI proposals: 260 training labels, 200 validation labels, and 200 test labels. The labels were categorized with assistance of AI and then independtly validated by a human.

Group assignments were fixed before label drafting. New candidates excluded the 229 groups exposed in earlier supervised work and diagnostics. Related passages stayed in the same split. Thirty-seven candidate passages were skipped and replaced within their assigned queues.

The splits had no overlapping passage IDs, review IDs, release groups, or casefolded exact texts. Normalized-text and word-trigram checks also found no cross-split matches at the specified threshold. These checks did not rule out all semantic similarity. The source corpus had already been used for unsupervised exploration.

Test data remained separate from training and validation until checkpoint selection and reload verification were complete.

In [3]:
authorized(ALLOW_DATA_UPLOAD)
import sys, importlib.metadata
assert not globals().get("SETUP_NEEDS_RESTART",False), "Restart after installation."
for name,pin in PINS.items():assert importlib.metadata.version(name)==pin, "Complete the pinned setup first."
from pathlib import Path
import hashlib,io,json,tempfile,zipfile
from google.colab import files
WORK_ROOT=Path(tempfile.mkdtemp(prefix="music_expanded_B_",dir="/content"))
EXPECTED_HELPERS={'expansion_contract.py': '7698b977b74ccaddc82e3a1fccd9b8671cee09e7445fc5f9bc64a8a77d542fc0', 'expansion_training.py': '3fd64240ead084f1b4d616a300f811a4b6140086f8208524d657140d1d28438e', 'expansion_preflight.py': '0d30c798f9b97406348e445dd47999d4c65ed45bfee2cd6c30ea12b286caf570', 'expansion_final_eval.py': '795c3348d7268d64e756dc6b93f02853a57aa8f7ece551a07ccbf22d6d2cad5e', 'requirements_colab_expansion.txt': '96299a5068219dbe61388da3ba34ccdf631e855f5e91292f0877c3cca015e018'}
COMMON={*EXPECTED_HELPERS,"active_candidate_manifest.json","frozen_manifest.json","review_evidence.json","package_receipt.json","package_sha256.json"}
def unpack_verified(uploaded,stage):
    assert len(uploaded)==1, "Choose exactly one stage-specific package."
    blob=next(iter(uploaded.values()))
    expected=COMMON|({"data/train.csv","data/validation.csv","old_training_140_unchanged.csv"} if stage=="train_validation" else {"data/test.csv"})
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        assert len(z.namelist())==len(expected) and set(z.namelist())==expected, "Wrong stage, duplicate or unexpected files."
        assert sum(i.file_size for i in z.infolist())<15_000_000, "Unexpected package size."
        contents={name:z.read(name) for name in expected}
    for name,digest in EXPECTED_HELPERS.items():assert hashlib.sha256(contents[name]).hexdigest()==digest, "Changed helper: "+name
    assert hashlib.sha256(contents["frozen_manifest.json"]).hexdigest()==EXPECTED_FREEZE_SHA256, "Wrong authorized freeze."
    hashes=json.loads(contents["package_sha256.json"])
    assert set(hashes)==expected-{"package_sha256.json"}
    for name,digest in hashes.items():assert hashlib.sha256(contents[name]).hexdigest()==digest, "Changed package member: "+name
    root=WORK_ROOT/stage;root.mkdir(exist_ok=False)
    for name,value in contents.items():
        target=root/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(value)
    return root
train_package=unpack_verified(files.upload(),"train_validation")
sys.path.insert(0,str(train_package))
import expansion_contract as contract
parts,manifest=contract.load_package(train_package,EXPECTED_FREEZE_SHA256)
print("Verified row counts:",{s:len(r) for s,r in parts.items()})
print("Training-only weights A/B/C:",contract.class_weights(parts["train"]))
print("Schedule:",contract.schedule(len(parts["train"])))
print("AI-assisted labels with human review; test not loaded or scored.")

Saving music_expansion_train_validation.zip to music_expansion_train_validation.zip
Verified row counts: {'train': 400, 'validation': 200}
Training-only weights A/B/C: [0.6116207951070336, 1.1594202898550725, 1.9900497512437811]
Schedule: {'batch_size': 2, 'gradient_accumulation': 4, 'epochs': 3, 'microbatches_per_epoch': 200, 'steps_per_epoch': 50, 'planned_optimizer_steps': 150, 'warmup_steps': 15}
AI-assisted labels with human review; test not loaded or scored.


## 3. Training and Checkpoint Selection

The study trained LoRA adapters while keeping the base model frozen. The fresh B-style run used rank 8, alpha 16, dropout 0.05, and the `q_proj` and `v_proj` layers. The learning rate was 0.0002 for three epochs.

The model learned to select the A, B, or C category at the final prompt position. The correct answer was not included in the input. Class weights were calculated from training labels only as N/(3 × class count).

| Category | Training passages | Weight |
|---|---:|---:|
| A — Specific musical observation | 218 | 0.611621 |
| B — General opinion | 115 | 1.159420 |
| C — Background/other | 67 | 1.990050 |

These weights increased the contribution of less frequent categories. Training used batches of two passages and accumulated gradients over four batches per update. Of 150 attempted updates, 148 succeeded and two were skipped by mixed-precision gradient scaling in the first epoch.

| Epoch | Validation accuracy | Validation macro-F1 | Unweighted cross-entropy |
|---|---:|---:|---:|
| 1 | 0.620 | 0.332114 | 0.969130 |
| 2 | 0.625 | 0.480290 | 0.852681 |
| 3 | 0.645 | 0.562990 | 0.781464 |

Epoch 3 was selected using validation macro-F1. The fixed tie-breaking rules used lower unweighted cross-entropy and then an earlier epoch. Test results played no role in selection. Parameter checks confirmed that the base weights stayed unchanged and the adapter weights changed.

In [4]:
authorized(ALLOW_TRAINING)
import torch
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0)
assert torch.__version__.split("+")[0]=="2.11.0"
from transformers import AutoTokenizer
import expansion_training as training
VERSIONS={n:importlib.metadata.version(n) for n in ["torch",*PINS]}
tokenizer=AutoTokenizer.from_pretrained(training.MODEL_ID,revision=training.MODEL_REVISION,trust_remote_code=False)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"
OUTPUT=WORK_ROOT/"expanded_results";OUTPUT.mkdir(exist_ok=False)
summary=training.run_experiment(contract.RUN_NAME,tokenizer,parts["train"],parts["validation"],manifest,OUTPUT,VERSIONS)
TRAINED_RUN=OUTPUT/contract.RUN_NAME
selection=contract.verify_selection(TRAINED_RUN,EXPECTED_FREEZE_SHA256)
print("Training finished. Validation-selected epoch:",selection["selected"]["epoch"])
print("Validation macro-F1:",selection["selected"]["validation"]["macro_f1"])
print("Test has not been loaded or scored.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

B_expanded_weighted_ABC epoch 1 validation macro-F1 0.332114 predictions {'specific_musical_observation': 187, 'general_opinion': 5, 'background_other': 8} updates/skips 48 2
B_expanded_weighted_ABC epoch 2 validation macro-F1 0.48029 predictions {'specific_musical_observation': 146, 'general_opinion': 26, 'background_other': 28} updates/skips 50 0
B_expanded_weighted_ABC epoch 3 validation macro-F1 0.56299 predictions {'specific_musical_observation': 114, 'general_opinion': 42, 'background_other': 44} updates/skips 50 0
Training finished. Validation-selected epoch: 3
Validation macro-F1: 0.5629897528631705
Test has not been loaded or scored.


## 4. Test Results and Limitations

The selected adapter and tokenizer were reloaded before test access. Tokenizer settings, prompt encodings, parameter fingerprints, and saved validation predictions passed verification. The same verified model then scored the test passages with adapters disabled for the unadapted baseline and enabled for the adapted model.

Each model scored the same 200 test passages once. The passages came from 99 review/release groups.

| Model | Test accuracy | Test macro-F1 |
|---|---:|---:|
| Unadapted base model | 31.5% | 0.238963 |
| Adapted epoch-3 model | 71.0% | 0.662959 |

The adapted model correctly classified 142 of 200 passages. On this test set, accuracy increased by 39.5 percentage points and macro-F1 increased by 0.423996.

| Category | Test support | Adapted recall |
|---|---:|---:|
| Specific musical observation | 113 | 79.6% |
| General opinion | 54 | 50.0% |
| Background/other | 33 | 75.8% |

General opinion had the weakest recall. The adapted model identified 27 of 54 general-opinion passages, compared with 90 of 113 musical observations and 25 of 33 background passages.

The paired comparison supported improved classification after adaptation on this test set. It did not establish that data expansion alone caused the gains. The prototype and expanded studies used different evaluation sets. No further training or checkpoint selection followed the final test.

The test passages were clustered within source groups. Subjective labels, approximate passage boundaries, the bounded review sample, and possible model pretraining exposure limited generalization. The restricted A/B/C scores were not calibrated confidence estimates. The study did not measure time savings in a practical workflow.

In [5]:
authorized(ALLOW_FINAL_TEST)
contract.verify_selection(TRAINED_RUN,EXPECTED_FREEZE_SHA256)  # before requesting test upload
assert not (TRAINED_RUN/"test_attempt_started.json").exists(), "Test attempt already recorded; stop."
import expansion_final_eval
prepared_final=expansion_final_eval.prepare_test_comparison(train_package,EXPECTED_FREEZE_SHA256,TRAINED_RUN,authorized=True)
print("Selected tokenizer/adapter preflight passed using training/validation only. Test has not been uploaded or opened.")

Selected tokenizer/adapter preflight passed using training/validation only. Test has not been uploaded or opened.


In [6]:
authorized(ALLOW_FINAL_TEST)
assert "prepared_final" in globals(), "Complete development-only preflight first."
test_package=unpack_verified(files.upload(),"test_after_selection")
comparison=expansion_final_eval.run_test_comparison(test_package,EXPECTED_FREEZE_SHA256,TRAINED_RUN,prepared_final,authorized=True)
for name in ["unadapted","adapted"]:
    print(name,"test macro-F1",comparison[name]["metrics"]["macro_f1"],"accuracy",comparison[name]["metrics"]["accuracy"])
print("Adapted minus unadapted macro-F1:",comparison["adapted_minus_unadapted_macro_f1"])
print("Final test complete. Do not tune, reselect or rerun based on these outcomes.")

Saving music_expansion_test_after_selection.zip to music_expansion_test_after_selection.zip
unadapted test macro-F1 0.2389626991736292 accuracy 0.315
adapted test macro-F1 0.6629586396067372 accuracy 0.71
Adapted minus unadapted macro-F1: 0.42399594043310795
Final test complete. Do not tune, reselect or rerun based on these outcomes.


## 5. Conclusion and AI Assistance

The adapted model performed better than the unadapted baseline on the paired test. General opinions remained difficult to identify. Future research could use more varied training examples and clearer guidelines for passages that combined opinion, musical detail, and background information. A new study would require a new untouched test set.

Saved predictions were used to verify the reported accuracy, macro-F1, class metrics, and confusion matrices. The results retained the training configuration, validation history, checkpoint selection, reload checks, and final test predictions.

### AI Assistance

Codex assisted and provided guidance for coding, label proposals, training helpers, browser-based execution, result verification, and the revision for written explanations. Aaron wrote the orginal code, reviewed all 660 new usable labels, and approved evaluation metrics and project purpose. The recorded contributions did not establish independent human annotation or independent validation.

In [7]:
# Creates a ZIP in this runtime only. No automatic upload/publication/submission.
import shutil
result_archive=shutil.make_archive(str(WORK_ROOT/"music_expanded_B_actual_results"),"zip",root_dir=OUTPUT)
print("Download using Colab's Files pane:",result_archive)
print("Also download the executed notebook. Preserve any failed-attempt evidence.")

Download using Colab's Files pane: /content/music_expanded_B_0ptktxny/music_expanded_B_actual_results.zip
Also download the executed notebook. Preserve any failed-attempt evidence.
